# Exercise 1: Case suffix analysis in NER

*Sequence Labelling · version 0.1.0*

> Take the KazNERD test set. For every GPE, LOCATION and PERSON mention, analyse the final word with the Apertium analyser in TurkicNLP and record its case. Keep three outcomes apart: a non-nominative case, nominative, and *unknown* (no analysis, or several readings with different cases) — do not count analyser failures as nominative. Manually check a random sample of 50 mentions and report the analyser's error rate on it. What proportion of mentions of each type is inflected? Is a trained model's recall (for example that of the pretrained Stanza KazNERD model) lower for some cases?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "stanza==1.11.0" sklearn-crfsuite transformers sentencepiece requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`, which the first cell uses instead). Stanza is pinned to 1.11.0 because the worked solutions use its Kazakh KazNERD checkpoint; models already on disk are used from `STANZA_RESOURCES_DIR` (default `~/stanza_resources`), and missing ones are downloaded. The datasets are read from commit-pinned GitHub files (KazNERD, KyrgyzNER, the Starlang Turkish NER data, TRopBank, the Turkish WordNet and UD 2.17 IMST), downloaded once into `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`) and checked against SHA-256 hashes. Taggers receive the **gold words**, so tokenisation errors do not enter the scores.

**Definitions.** The case of a mention is read from all Apertium readings of its final word, ignoring copula continuations (`+е<cop>…`). Nominative and the attributive form (`<attr>`, a caseless modifier use) are both counted as *unmarked*. If the readings agree on one case, that case is recorded; if there is no analysis, or the readings disagree, the outcome is *unknown*. Recall is strict: the Stanza model must predict the gold span and type exactly. The model receives the gold words of every test sentence.

In [ ]:
import collections, hashlib, os, random, re, statistics
from pathlib import Path
import requests

# Pinned public releases (commit-addressed raw files) and their SHA-256 hashes.
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_{}.txt"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/data/{}"
SHA256 = {
    KAZNERD.format("train"): "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KAZNERD.format("valid"): "29d584d1a0d9aca6d58b39eb8a93e154791d303ecff56ef93f12b3c889239bf1",
    KAZNERD.format("test"): "359a1f61f6a21280c6c47b4fde9a5adcd07411d62015aa6e32d940c51f163b7a",
    KYRGYZNER.format("KyrgyzNER_TRAIN.jsonl.txt"): "719a74a18cd99d6709c1e0592d0337aafade65aa97fe762e62f3f72a6b343a26",
    KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"): "57a35a92139497819a2792d3e9e00a41b387b029b7851be7a64f4968befaddf5",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()

def fetch(url, sha256=None):
    """Download a pinned file once, check its hash and return the local path."""
    sha256 = sha256 or SHA256[url]
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha256:
        raise ValueError(f"{url} differs from the pinned release")
    return path

def read_conll(path, tag_col=-1, comment="#"):
    """Sentences as (words, tags) from a column file with blank-line sentence breaks."""
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words:
                sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith(comment + " "):
            cols = line.split("\t") if "\t" in line else line.split()
            words.append(cols[0]); tags.append(cols[tag_col].strip())
    return sents

def spans(tags):
    """Strict IOB2 spans {(start, end, type)}; an I- tag that does not continue a span starts none."""
    out, start, typ = set(), None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if t.startswith("I-") and typ == t[2:]:
            continue
        if typ is not None:
            out.add((start, i, typ))
        start, typ = (i, t[2:]) if t.startswith("B-") else (None, None)
    return out

def bioes_to_iob2(tags):
    return ["B-" + t[2:] if t[:2] in ("S-", "B-") else "I-" + t[2:] if t[:2] in ("E-", "I-") else "O" for t in tags]

def prf(gold, pred):
    """Micro precision, recall and F1 (%) over lists of span sets."""
    g = sum(len(x) for x in gold); p = sum(len(x) for x in pred)
    c = sum(len(x & y) for x, y in zip(gold, pred))
    pr = c / p if p else 0.0; rc = c / g if g else 0.0
    return {"gold": g, "pred": p, "correct": c, "P": round(100 * pr, 2), "R": round(100 * rc, 2),
            "F1": round(100 * (2 * pr * rc / (pr + rc) if pr + rc else 0.0), 2)}

def stanza_ner(lang="kk"):
    """Stanza NER over gold words; models from STANZA_RESOURCES_DIR, downloaded only if missing."""
    import logging, stanza
    logging.getLogger("stanza").setLevel(logging.ERROR)
    model_dir = os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser()))
    have = (Path(model_dir) / lang / "ner").exists()
    return stanza.Pipeline(lang, processors="tokenize,ner", tokenize_pretokenized=True, model_dir=model_dir,
                           download_method=None if have else "reuse_resources", verbose=False)

def tag_words(nlp, sents, batch=2000):
    """IOB2 predictions for gold-word sentences (spaces inside a token are joined with '_')."""
    out = []
    for i in range(0, len(sents), batch):
        doc = nlp([[w.replace(" ", "_") for w in words] for words, _ in sents[i:i + batch]])
        for s, (words, _) in zip(doc.sentences, sents[i:i + batch]):
            assert len(s.tokens) == len(words)
            out.append(bioes_to_iob2([t.ner for t in s.tokens]))
    return out

In [ ]:
import time
try:
    import hfst
except ImportError:
    import sys, pyhfst
    sys.modules["hfst"] = pyhfst
import turkicnlp
from turkicnlp import Pipeline

turkicnlp.download("kaz", processors=["morph"])            # Apertium Kazakh transducer (once)
kaz = Pipeline("kaz", processors=["tokenize", "morph"], tokenize_backend="rule", morph_backend="apertium")
analyser = next(p for p in kaz._processors if hasattr(p, "_analyze"))   # private API in 0.5.0
CASES = {"nom", "gen", "dat", "acc", "loc", "abl", "ins"}

def case_of(word):
    """'unmarked' (nom or attributive), a case name, or 'unknown' (no analysis, or readings that
    disagree on the case). Copula continuations ('+е<cop>...') are ignored."""
    found = set()
    for r in analyser._analyze(word):
        raw = r["raw"].split("+")[0]
        if raw.startswith("*") or "<" not in raw:
            continue
        tags = set(re.findall(r"<([^>]+)>", raw))
        found |= {"unmarked" if c == "nom" else c for c in tags & CASES} | ({"unmarked"} if "attr" in tags else set())
    return found.pop() if len(found) == 1 else "unknown"

test = read_conll(fetch(KAZNERD.format("test")))
t0 = time.time()
pred = tag_words(stanza_ner("kk"), test)
print(f"Stanza KazNERD model on {len(test)} test sentences ({time.time() - t0:.0f} s):",
      prf([spans(t) for _, t in test], [spans(p) for p in pred]))

TYPES = ("GPE", "LOCATION", "PERSON")
rows = []                                    # (type, case of final word, recalled?, mention)
for (words, tags), p in zip(test, pred):
    ps = spans(p)
    for s, e, typ in spans(tags):
        if typ in TYPES:
            rows.append((typ, case_of(words[e - 1]), (s, e, typ) in ps, " ".join(words[s:e])))

print("\ncase of the final word, share of mentions (n):")
order = ["unmarked", "gen", "dat", "acc", "loc", "abl", "ins", "unknown"]
for typ in TYPES:
    c = collections.Counter(case for t, case, *_ in rows if t == typ); n = sum(c.values())
    print(f"  {typ:9} n={n:5}  " + "  ".join(f"{k} {100 * c[k] / n:.1f}%" for k in order if c[k]))
print("\nrecall of the Stanza model by case (all three types), cases with n >= 20:")
for case in order:
    v = [ok for _, c, ok, _ in rows if c == case]
    if len(v) >= 20:
        print(f"  {case:9} n={len(v):5}  recall {100 * sum(v) / len(v):.1f}%")

random.seed(9)
print("\nrandom sample of 50 mentions for a manual check of the analyser:")
sample = random.sample(rows, 50)
for typ, case, ok, m in sample:
    print(f"  {typ:9} {case:9} {m}")

## Worked solution

Output of `experiments/ch09_exercises/run_ch09_exercises.py` (KazNERD release `bd4333d`; Stanza 1.11.0, KazNERD checkpoint):

```
Stanza KazNERD model on 11307 test sentences (247 s): {'gold': 13508, 'pred': 13501, 'correct': 12821, 'P': 94.96, 'R': 94.91, 'F1': 94.94}

case of the final word, share of mentions (n):
  GPE       n= 1742  unmarked 40.0%  gen 8.4%  dat 4.2%  acc 1.0%  loc 19.1%  abl 6.8%  ins 1.1%  unknown 19.3%
  LOCATION  n=  212  unmarked 39.2%  gen 9.4%  dat 7.1%  acc 1.9%  loc 5.2%  abl 2.4%  ins 1.4%  unknown 33.5%
  PERSON    n= 1332  unmarked 34.9%  gen 8.2%  dat 1.5%  acc 1.6%  abl 0.5%  ins 1.3%  unknown 52.1%

recall of the Stanza model by case (all three types), cases with n >= 20:
  unmarked  n= 1244  recall 96.1%
  gen       n=  275  recall 98.9%
  dat       n=  108  recall 95.4%
  acc       n=   43  recall 97.7%
  loc       n=  344  recall 98.0%
  abl       n=  130  recall 97.7%
  ins       n=   40  recall 95.0%
  unknown   n= 1102  recall 94.6%

random sample of 50 mentions for a manual check of the analyser:
  GPE       unmarked  Қытай
  LOCATION  unmarked  Кіші Арал
  PERSON    unknown   Индира Жылқайдарова
  PERSON    unknown   Сооронбай Жээнбековпен
  PERSON    gen       Ерлан Нұрғалиевтің
  GPE       gen       Түркістан облысының
  PERSON    dat       Арманға
  PERSON    unknown   Наталья Бестемьянова
  GPE       unmarked  Қазақстан
  GPE       gen       Қазақстанның
  PERSON    unknown   Іңкәрдің
  PERSON    unknown   Ердәулет Байдуллаев
  GPE       unmarked  Грекия
  PERSON    unknown   Айлин
  GPE       unknown   Қапчағайда
  PERSON    unmarked  Қасым-Жомарт Тоқаев
  GPE       unmarked  Қазақстан
  GPE       unknown   Алматыдағы
  PERSON    unmarked  Аян Сәрсенбаев
  GPE       unmarked  Қазақстан
  PERSON    unknown   Динара Балғазина
  GPE       unknown   Түркістандағы
  GPE       loc       Түркістан облысында
  GPE       gen       Ресейдің
  GPE       abl       Алматыдан
  PERSON    gen       Құнанбайдың
  PERSON    unknown   Бақыт Топтаева
  PERSON    unknown   Ербол Дайыров
  PERSON    unmarked  Иван Дүйсенов
  GPE       unmarked  Қазақстан
  PERSON    unmarked  Жайдар Омаров
  GPE       unknown   Атырау облысындағы
  GPE       unmarked  Өзбекстан
  GPE       loc       Алматыда
  GPE       unmarked  Қазақстан
  GPE       unmarked  Түркия
  PERSON    unknown   Эдуард Кон
  PERSON    unmarked  Әлия Молдағұлова
  GPE       unmarked  Қытай
  PERSON    unmarked  Қасым-Жомарт Тоқаев
  PERSON    unknown   Сергей Трахтенберг
  GPE       loc       Ресейде
  PERSON    ins       Нұрсұлтан Назарбаевпен
  GPE       unmarked  Алматы
  PERSON    unmarked  Арыстанбек Кенже
  LOCATION  acc       Аралды
  GPE       unmarked  Киров
  PERSON    unknown   Зейін
  PERSON    gen       Ермек Серкебаевтың
  PERSON    unmarked  Димаш Құдайберген
```

**The model.** On the whole test set the pretrained Stanza model reaches strict micro F1 94.94 (13,508 gold entities, 12,821 correct), the same score as the chapter's baseline table.

**How often mentions are inflected.** The percentages use all gold mentions of each type as the denominator. For GPE, 40.0% are unmarked and 19.3% unknown; about 40.7% receive an overt case label, including locative (19.1%), genitive (8.4%) and ablative (6.8%). LOCATION mentions are unmarked in 39.2% of cases and unknown in 33.5%. These analyser-derived groups require the manual check below. PERSON mentions are the hardest to measure: the analyser has no analysis, or conflicting ones, for 52.1% of them, with unknown names among the examples (*Индира Жылқайдарова*, *Наталья Бестемьянова*). *Іңкәрдің* in the sample is clearly a genitive, but it is counted as unknown, as the exercise requires. Counting the unknowns as nominative would have reported 87% of PERSON mentions as uninflected.

**Recall by case.** Recall is high for every case: from 95.0% (instrumental) to 98.9% (genitive), with 96.1% for unmarked mentions and 94.6% for the unknown group. These observational groups do not isolate the effect of adding a case suffix: entity strings, contexts and support differ. Training-set membership of the individual inflected forms was not established. The unknown group has lower recall than the unmarked group in this sample.

**The manual check.** The sample of 50 mentions printed above is the input for the manual check. Where the analyser assigns a case, the labels shown match the suffixes (for example *Нұрсұлтан Назарбаевпен* `ins`, *Түркістан облысының* `gen`, *Арманға* `dat`, *Аралды* `acc`). The *unknown* lines contain two kinds of word. Some are unknown names that are clearly inflected (*Іңкәрдің* genitive, *Сооронбай Жээнбековпен* instrumental). Others are locative forms with the relational suffix *-ғы/-гі* (*Алматыдағы*, *Түркістандағы*, *Атырау облысындағы* 'in/of Almaty …'), for which the analyser returns readings that disagree on case. The reader should check every line and count errors separately for the case analysis and for the *unknown* decisions.

## Your turn — extension

1. Repeat with a model whose training data contain fewer inflected forms (for example zero-shot on KyrgyzNER, Exercise 3). Does recall now differ by case?
2. Add a suffix-rule fallback for unknown names (*-ның/-нің/-дың…* genitive, *-да/-де/-та/-те* locative …) and measure on the manual sample how often it is right.